# Episode 19: Pretraining on the web: a GPT overnight

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep019-web.ipynb)

We pretrain a model in GPT-2 small's shape on educational web pages for a whole night, on one RTX 4090 rented for under four dollars, and measure it against GPT-2 small on HellaSwag.

[The video](https://youtu.be/15vafg1wdpI) · [The episode's page](https://mvgpt.nenuadrian.com/ep019-web/) · [The docs](https://nenuadrian.github.io/mvGPT/web/) · [The code at `ep019/end`](https://github.com/nenuadrian/mvGPT/tree/ep019/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep019/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt
!python setup.py -q build_ext --inplace  # the C++ BPE tokenizer, as a Python module

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))

## The data
The stories' merges first, for a comparison, then FineWeb-Edu's.

In [ ]:
!python data.py --bpe  # TinyStories, the BPE merges, and the stories as BPE token ids
!python data.py --web 1  # FineWeb-Edu: two files of about 2 GB, the web merges, the held-out pages and a file of tokens

In [ ]:
from data import load_tokenizer

text = open("data/valid-web.txt").read(2_000_000)
for m in ("merges.txt", "web-merges.txt"):
    print(f"{m:<16}{len(text) / len(load_tokenizer(True, m).encode(text)):.2f} characters per token")

## A night on one GPU
The episode's run: 470 minutes on a rented RTX 4090, on five files of FineWeb-Edu
(`python data.py --web 5`):

```
torchrun --standalone --nproc_per_node 1 pretrain_ddp.py 470 --web --compile --block 1024 --batch 24 --accum 4
```

Here are ten minutes of it on Colab's GPU and one file, in batches of 8, to
`checkpoints/web-colab`.

In [ ]:
import torch

# bfloat16 in hardware: an L4 or A100 runtime has it, the free T4 (compute capability 7.5) doesn't.
AMPERE = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
print("bfloat16 in hardware" if AMPERE else "no bfloat16 in hardware: these runs will be slow")

In [ ]:
COMPILE = "--compile" if AMPERE else ""
!torchrun --standalone --nproc_per_node 1 pretrain_ddp.py 10 --web {COMPILE} --block 1024 --batch 8 --accum 4 --name web-colab

## The model from the night
The episode's, from the Hub, against episode 12's model of the stories.

In [ ]:
fetch("pretrain-web", "pretrain-ddp")

In [ ]:
!python sample.py checkpoints/pretrain-web/model.pt "Photosynthesis is" -n 1 --tokens 200 --seed 1
!python sample.py checkpoints/pretrain-web/model.pt "The French Revolution" -n 1 --tokens 120 --seed 2
!python sample.py checkpoints/pretrain-web/model.pt "To grow tomatoes at home," -n 1 --tokens 120 --seed 3
!python sample.py checkpoints/pretrain-ddp/model.pt "To grow tomatoes at home," -n 1 --tokens 120 --seed 3

## HellaSwag
GPT-2 small, the stories model, and the web model.

In [ ]:
!python hellaswag.py gpt2 2>/dev/null
!python hellaswag.py checkpoints/pretrain-ddp/model.pt
!python hellaswag.py checkpoints/pretrain-web/model.pt

## Bits per character, on stories and on web pages

In [ ]:
!python bits.py checkpoints/pretrain-ddp/model.pt checkpoints/pretrain-web/model.pt
!python bits.py checkpoints/pretrain-ddp/model.pt checkpoints/pretrain-web/model.pt --web

Next: [episode 20, LLM tool calling from scratch: a calculator](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep020-tools.ipynb)